In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
import torch
import cv2
import numpy as np
from transformers import CLIPProcessor

# ==================================
# Step 1: Setup and Imports
# ==================================

# Add your source code directory to the Python path
import sys
project_path = '/content/drive/My Drive/H-PeVL' # <-- Make sure this is correct
sys.path.append(f'{project_path}/src')

# Import your model class
from model import HPeVLModel

In [28]:
# ==================================
# Step 2: Define the Preprocessing Function
# ==================================

def preprocess_video_for_inference(video_path, processor, num_frames=16):
    """Takes a video file path and prepares it for the model."""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)

    frames = []
    for i in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame_rgb)
    cap.release()

    inputs = processor(images=frames, return_tensors="pt")
    return inputs["pixel_values"].unsqueeze(0) # Add a batch dimension


In [29]:

# ==================================
# Step 3: Load the Trained Model
# ==================================

device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint_path = f"{project_path}/checkpoints_v2/final_best_model.pth"

# 1. Load the model architecture (the "blueprint")
model = HPeVLModel().to(device)

# 2. Load the saved weights (the "learned knowledge")
model.load_state_dict(torch.load(checkpoint_path))

# 3. Set the model to evaluation mode (IMPORTANT!)
model.eval()

print("✓ MEpoch 1 finished. Avg l loaded successfully and set to evaluation mode.")


✓ CLIP parameters frozen
✓ Full UCMR Block initialized
✓ H-PeVL Model Initialized (dim=256)
✓ MEpoch 1 finished. Avg l loaded successfully and set to evaluation mode.


In [30]:
!pip install mediapipe

In [34]:
# ==================================
# Generate Pose Data for New Video
# ==================================
import cv2
import mediapipe as mp
import numpy as np
import os

# 1. Setup Paths
video_path = f"{project_path}/data/Test/charge.mp4"
output_pose_path = f"{project_path}/data/Test/charge_skeleton.txt"

print(f"Processing video: {video_path}")

# 2. Initialize MediaPipe Hands
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5
)

cap = cv2.VideoCapture(video_path)
frame_count = 0
pose_data_lines = []

if not cap.isOpened():
    print("Error: Could not open video.")
else:
    print("Extracting skeleton data... (this might take a minute)")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert BGR to RGB
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(image_rgb)

        # Format: FrameIndex x0 y0 z0 x1 y1 z1 ... x20 y20 z20
        # We start the line with the frame index
        line_data = [f"{frame_count:04d}"]

        if results.multi_hand_landmarks:
            # Get the first hand detected
            hand_landmarks = results.multi_hand_landmarks[0]

            # MediaPipe returns normalized coordinates (0-1).
            # We scale them to pixel coordinates to match your training data format
            h, w, c = frame.shape

            for landmark in hand_landmarks.landmark:
                cx, cy = landmark.x * w, landmark.y * h
                # Z is relative depth, we scale it similarly or keep as is.
                # Looking at your sample data (values like 348.0), it seems Z is also scaled.
                cz = landmark.z * w

                line_data.append(f"{cx:.6f}")
                line_data.append(f"{cy:.6f}")
                line_data.append(f"{cz:.6f}")
        else:
            # If no hand found, fill with zeros to keep shape consistent
            # (21 points * 3 coords = 63 zeros)
            line_data.extend(["0.000000"] * 63)

        # Join into a single string line
        pose_data_lines.append(" ".join(line_data))
        frame_count += 1

    cap.release()
    hands.close()

    # 3. Save to file
    with open(output_pose_path, "w") as f:
        f.write("\n".join(pose_data_lines))

    print(f"✓ Successfully saved skeleton file to: {output_pose_path}")
    print(f"  Total frames processed: {frame_count}")

Processing video: /content/drive/My Drive/H-PeVL/data/Test/charge.mp4
Extracting skeleton data... (this might take a minute)
✓ Successfully saved skeleton file to: /content/drive/My Drive/H-PeVL/data/Test/charge_skeleton.txt
  Total frames processed: 185


In [35]:
# ==================================
# Step 4: Prepare Inputs and Make a Prediction (H-PeVL Version)
# ==================================
import torch
import torch.nn.functional as F
import numpy as np
import cv2
from transformers import CLIPProcessor

# 1. Initialize Processor (This fixes your NameError)
print("Loading CLIP processor...")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")

# 2. Define Helper Functions
def preprocess_video_for_inference(video_path, processor, num_frames=16):
    """Reads a video, samples frames, and prepares them for the model."""
    cap = cv2.VideoCapture(video_path)
    frames = []
    if not cap.isOpened():
        print(f"Error opening video: {video_path}")
        return None

    # Read all frames
    all_frames = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        all_frames.append(frame)
    cap.release()

    # Sample frames (uniform sampling)
    total_frames = len(all_frames)
    if total_frames == 0: return None
    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)

    sampled_frames = []
    for i in indices:
        frame_rgb = cv2.cvtColor(all_frames[i], cv2.COLOR_BGR2RGB)
        sampled_frames.append(frame_rgb)

    # Process using CLIP processor
    inputs = processor(images=sampled_frames, return_tensors="pt")
    # Add batch dimension: (1, num_frames, 3, H, W)
    return inputs['pixel_values'].unsqueeze(0)

def preprocess_pose_for_inference(pose_path, num_frames=16):
    """Loads and processes a single pose file for inference."""
    try:
        all_pose_data = np.loadtxt(pose_path)
        total_frames = all_pose_data.shape[0]
        indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
        sampled_pose = all_pose_data[indices]
        # Add batch dimension: (1, 16, 64)
        return torch.from_numpy(sampled_pose).float().unsqueeze(0)
    except Exception as e:
        print(f"Error loading pose file: {e}")
        return None

# --- Main Inference Setup ---

# 3. SETUP PATHS
# Make sure these point to your specific test files
new_video_path = f"{project_path}/data/Test/charge.mp4"
new_pose_path  = f"{project_path}/data/Test/charge_skeleton.txt"

# 4. DEFINE ACTIONS
all_possible_actions = [
    'charge_cell_phone', 'clean_glasses', 'close_juice_bottle', 'close_liquid_soap',
    'close_milk', 'close_peanut_butter', 'drink_mug', 'flip_pages', 'flip_sponge',
    'give_card', 'give_coin', 'handshake', 'high_five', 'light_candle',
    'open_juice_bottle', 'open_letter', 'open_liquid_soap', 'open_milk',
    'open_peanut_butter', 'open_soda_can', 'open_wallet', 'pour_juice_bottle',
    'pour_liquid_soap', 'pour_milk', 'pour_wine', 'prick', 'put_salt', 'put_sugar',
    'put_tea_bag', 'read_letter', 'receive_coin', 'scoop_spoon', 'scratch_sponge',
    'sprinkle', 'squeeze_paper', 'squeeze_sponge', 'stir', 'take_letter_from_enveloppe',
    'tear_paper', 'toast_wine', 'unfold_glasses', 'use_calculator', 'use_flash',
    'wash_sponge', 'write'
]

# 5. PREPARE INPUTS
# A. Video
print(f"Processing video: {new_video_path}")
video_tensor = preprocess_video_for_inference(new_video_path, processor).to(device)

# B. Pose
print(f"Processing pose: {new_pose_path}")
pose_tensor = preprocess_pose_for_inference(new_pose_path)

if pose_tensor is None:
    raise FileNotFoundError("Could not load pose data. Inference cannot continue without pose.")
pose_tensor = pose_tensor.to(device)

# C. Text
print("Processing text labels...")
clean_actions = [f"a video of a hand performing the action of {action.replace('_', ' ')}" for action in all_possible_actions]
text_inputs = processor(
    text=clean_actions,
    return_tensors="pt",
    padding=True
).to(device)

# 6. RUN INFERENCE
print("Running H-PeVL inference...")
model.eval()
with torch.no_grad():
    # A. Forward Pass
    refined_video, refined_pose, text_features, _ = model(
        pixel_values=video_tensor,
        pose_values=pose_tensor,
        input_ids=text_inputs["input_ids"],
        attention_mask=text_inputs["attention_mask"]
    )

    # B. Normalize & Fuse
    refined_video = F.normalize(refined_video, dim=-1)
    refined_pose = F.normalize(refined_pose, dim=-1)
    text_features = F.normalize(text_features, dim=-1)

    fused_features = (refined_video + refined_pose) / 2.0
    fused_features = F.normalize(fused_features, dim=-1)

    # C. Calculate Similarity
    logit_scale = model.clip.logit_scale.exp()
    similarity_scores = (logit_scale * fused_features @ text_features.T)
    similarity_probs = similarity_scores.softmax(dim=-1)

# 7. SHOW RESULTS
top_probs, top_indices = similarity_probs.topk(5)

print("\n=== Predictions ===")
for i in range(5):
    idx = top_indices[0][i].item()
    score = top_probs[0][i].item()
    action_name = all_possible_actions[idx]
    print(f"{i+1}. {action_name}: {score:.4f}")


Loading CLIP processor...
Processing video: /content/drive/My Drive/H-PeVL/data/Test/charge.mp4
Processing pose: /content/drive/My Drive/H-PeVL/data/Test/charge_skeleton.txt
Processing text labels...
Running H-PeVL inference...

=== Predictions ===
1. flip_pages: 0.1449
2. write: 0.1120
3. take_letter_from_enveloppe: 0.0934
4. handshake: 0.0751
5. receive_coin: 0.0606


In [36]:

# ==================================
# Step 5: Interpret the Results
# ==================================

# Find the highest probability and its corresponding index
best_prob, best_idx = similarity_probs[0].max(dim=0)
predicted_action = all_possible_actions[best_idx]

print("\n--- Prediction Results ---")
print(f"Predicted Action: '{predicted_action}'")
print(f"Confidence: {best_prob.item():.2%}")


--- Prediction Results ---
Predicted Action: 'flip_pages'
Confidence: 14.49%
